In [1]:
import os
import pickle
import warnings
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    classification_report
)

warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", ConvergenceWarning)

# Create the mandatory 'model/' directory
os.makedirs("model", exist_ok=True)

In [5]:
import os
import urllib.request
import pandas as pd

# Download adult.data if it does not already exist locally
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
local_file = "adult.csv"

if not os.path.exists(local_file):
    print("Downloading adult.data from UCI repository...")
    try:
        urllib.request.urlretrieve(url, local_file)
        print("Download complete.")
    except Exception as e:
        print(f"Direct download failed ({e}). Please upload 'adult.data' manually to your working directory.")

columns = [
    "age", "workclass", "fnlwgt", "education",
    "education_num", "marital_status",
    "occupation", "relationship", "race",
    "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income"
]

df = pd.read_csv(local_file, names=columns, skipinitialspace=True)
print("Initial Shape:", df.shape)
df.head()

Initial Shape: (32561, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [6]:
# Check missing '?' counts and drop rows containing missing values cleanly
print("Missing '?' counts per column:\n", (df == "?").sum())

df = df.replace("?", pd.NA).dropna().reset_index(drop=True)
print("\nCleaned Shape:", df.shape)

Missing '?' counts per column:
 age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     583
income               0
dtype: int64

Cleaned Shape: (30162, 15)


In [7]:
# Map income categories: <=50K -> 0, >50K -> 1
df["income"] = df["income"].map({
    "<=50K": 0,
    ">50K": 1
})

df["income"].value_counts()

income
0    22654
1     7508
Name: count, dtype: int64

In [8]:
import pickle
from sklearn.preprocessing import LabelEncoder

X = df.drop("income", axis=1).copy()
y = df["income"].astype(int)

encoders = {}
for col in X.columns:
    if pd.api.types.is_string_dtype(X[col]) or X[col].dtype == "object":
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        encoders[col] = le

os.makedirs("model", exist_ok=True)
with open("model/label_encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nEncoded Feature Dtypes:\n", X.dtypes)

Features shape: (30162, 14)
Target shape: (30162,)

Encoded Feature Dtypes:
 age               int64
workclass         int64
fnlwgt            int64
education         int64
education_num     int64
marital_status    int64
occupation        int64
relationship      int64
race              int64
sex               int64
capital_gain      int64
capital_loss      int64
hours_per_week    int64
native_country    int64
dtype: object


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Export test_data.csv with both features and target ('income') for Streamlit evaluation
test_data = X_test.copy()
test_data["income"] = y_test.values
test_data.to_csv("test_data.csv", index=False)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Saved test_data.csv shape (with target):", test_data.shape)

X_train shape: (24129, 14)
X_test shape: (6033, 14)
Saved test_data.csv shape (with target): (6033, 15)


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print("Logistic Regression Metrics")
print("---------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"MCC      : {mcc:.4f}")

Logistic Regression Metrics
---------------------------
Accuracy : 0.8175
AUC      : 0.8501
Precision: 0.7135
Recall   : 0.4461
F1 Score : 0.5490
MCC      : 0.4613


In [11]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=10, min_samples_split=10, random_state=42)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)
dt_prob = dt.predict_proba(X_test)[:, 1]

dt_accuracy = accuracy_score(y_test, dt_pred)
dt_auc = roc_auc_score(y_test, dt_prob)
dt_precision = precision_score(y_test, dt_pred)
dt_recall = recall_score(y_test, dt_pred)
dt_f1 = f1_score(y_test, dt_pred)
dt_mcc = matthews_corrcoef(y_test, dt_pred)

print("Decision Tree Metrics")
print("---------------------")
print(f"Accuracy : {dt_accuracy:.4f}")
print(f"AUC      : {dt_auc:.4f}")
print(f"Precision: {dt_precision:.4f}")
print(f"Recall   : {dt_recall:.4f}")
print(f"F1 Score : {dt_f1:.4f}")
print(f"MCC      : {dt_mcc:.4f}")

Decision Tree Metrics
---------------------
Accuracy : 0.8505
AUC      : 0.8892
Precision: 0.7504
Recall   : 0.5985
F1 Score : 0.6659
MCC      : 0.5772


In [12]:
from sklearn.neighbors import KNeighborsClassifier

knn = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(n_neighbors=7))
])
knn.fit(X_train, y_train)

knn_pred = knn.predict(X_test)
knn_prob = knn.predict_proba(X_test)[:, 1]

knn_accuracy = accuracy_score(y_test, knn_pred)
knn_auc = roc_auc_score(y_test, knn_prob)
knn_precision = precision_score(y_test, knn_pred)
knn_recall = recall_score(y_test, knn_pred)
knn_f1 = f1_score(y_test, knn_pred)
knn_mcc = matthews_corrcoef(y_test, knn_pred)

print("KNN Metrics")
print("---------------------")
print(f"Accuracy : {knn_accuracy:.4f}")
print(f"AUC      : {knn_auc:.4f}")
print(f"Precision: {knn_precision:.4f}")
print(f"Recall   : {knn_recall:.4f}")
print(f"F1 Score : {knn_f1:.4f}")
print(f"MCC      : {knn_mcc:.4f}")

KNN Metrics
---------------------
Accuracy : 0.8221
AUC      : 0.8614
Precision: 0.6631
Recall   : 0.5806
F1 Score : 0.6191
MCC      : 0.5056


In [13]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)
nb_prob = nb.predict_proba(X_test)[:, 1]

nb_accuracy = accuracy_score(y_test, nb_pred)
nb_auc = roc_auc_score(y_test, nb_prob)
nb_precision = precision_score(y_test, nb_pred)
nb_recall = recall_score(y_test, nb_pred)
nb_f1 = f1_score(y_test, nb_pred)
nb_mcc = matthews_corrcoef(y_test, nb_pred)

print("Naive Bayes Metrics")
print("---------------------")
print(f"Accuracy : {nb_accuracy:.4f}")
print(f"AUC      : {nb_auc:.4f}")
print(f"Precision: {nb_precision:.4f}")
print(f"Recall   : {nb_recall:.4f}")
print(f"F1 Score : {nb_f1:.4f}")
print(f"MCC      : {nb_mcc:.4f}")

Naive Bayes Metrics
---------------------
Accuracy : 0.7865
AUC      : 0.8289
Precision: 0.6569
Recall   : 0.2983
F1 Score : 0.4103
MCC      : 0.3368


In [14]:
from sklearn.svm import SVC

svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(probability=True, random_state=42))
])
svm.fit(X_train, y_train)

svm_pred = svm.predict(X_test)
svm_prob = svm.predict_proba(X_test)[:, 1]

svm_accuracy = accuracy_score(y_test, svm_pred)
svm_auc = roc_auc_score(y_test, svm_prob)
svm_precision = precision_score(y_test, svm_pred)
svm_recall = recall_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred)
svm_mcc = matthews_corrcoef(y_test, svm_pred)

print("SVM Metrics")
print("---------------------")
print(f"Accuracy : {svm_accuracy:.4f}")
print(f"AUC      : {svm_auc:.4f}")
print(f"Precision: {svm_precision:.4f}")
print(f"Recall   : {svm_recall:.4f}")
print(f"F1 Score : {svm_f1:.4f}")
print(f"MCC      : {svm_mcc:.4f}")

SVM Metrics
---------------------
Accuracy : 0.8414
AUC      : 0.8888
Precision: 0.7466
Recall   : 0.5493
F1 Score : 0.6329
MCC      : 0.5450


In [16]:
results = pd.DataFrame({
    "ML Model Name": ["Logistic Regression", "Decision Tree", "KNN", "Naive Bayes", "Support Vector Machine"],
    "Accuracy": [accuracy, dt_accuracy, knn_accuracy, nb_accuracy, svm_accuracy],
    "AUC": [auc, dt_auc, knn_auc, nb_auc, svm_auc],
    "Precision": [precision, dt_precision, knn_precision, nb_precision, svm_precision],
    "Recall": [recall, dt_recall, knn_recall, nb_recall, svm_recall],
    "F1": [f1, dt_f1, knn_f1, nb_f1, svm_f1],
    "MCC": [mcc, dt_mcc, knn_mcc, nb_mcc, svm_mcc],
}).round(4)

results.to_csv("metrics.csv", index=False)
results

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.8175,0.8501,0.7135,0.4461,0.5490,0.4613
1,Decision Tree,0.8505,0.8892,0.7504,0.5985,0.6659,0.5772
2,KNN,0.8221,0.8614,0.6631,0.5806,0.6191,0.5056
3,Naive Bayes,0.7865,0.8289,0.6569,0.2983,0.4103,0.3368
4,Support Vector Machine,0.8414,0.8888,0.7466,0.5493,0.6329,0.5450


In [17]:
model_files = {
    "model/logistic_regression.pkl": lr,
    "model/decision_tree.pkl": dt,
    "model/knn.pkl": knn,
    "model/naive_bayes.pkl": nb,
    "model/svm.pkl": svm
}

for filepath, model_obj in model_files.items():
    with open(filepath, "wb") as f:
        pickle.dump(model_obj, f)

print("All models saved successfully in 'model/':", os.listdir("model"))

All models saved successfully in 'model/': ['label_encoders.pkl', 'logistic_regression.pkl', 'decision_tree.pkl', 'knn.pkl', 'naive_bayes.pkl', 'svm.pkl']
